# LWSCE pilot on Colab

CE vs LWSCE on CIFAR100: each loss picks its learning rate on the validation set, then test accuracy is compared.

1. **Runtime → Change runtime type → GPU** (T4 is enough).
2. **Runtime → Run all.** Google Drive will ask for permission once.
3. Results and models are saved to Google Drive (`MyDrive/label_mask_loss_runs`). If the session disconnects, just **Run all** again: finished loss × lr combinations are skipped (a combination that was interrupted midway restarts from its first run).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"
print("torch", torch.__version__)

## 1. Experiment settings

In [ ]:
# where the code comes from: "github" clones BRANCH, "zip" asks you to upload a zip of the repo
CODE_SOURCE = "github"
REPO_URL = "https://github.com/machine-learning-is-easy/label_mask_loss.git"
BRANCH = "revert"

DATASET = "CIFAR100"
OPT_ALG = "ADAM"
LOSSES = ["CROSSENTROPY", "LWSCE"]
LRS = ["1e-3", "3e-4", "1e-4"]
RUNS = 3
EPOCH = 31          # training.py trains EPOCH - 1 epochs
ALPHA = 0.2

DRIVE_DIR = "/content/drive/MyDrive/label_mask_loss_runs"
CODE_DIR = "/content/label_mask_loss"

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(os.path.join(DRIVE_DIR, "result"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, "model"), exist_ok=True)

## 3. Get the code

In [ ]:
import os, shutil, subprocess, zipfile

if CODE_SOURCE == "github":
    if os.path.exists(CODE_DIR):
        subprocess.run(["git", "-C", CODE_DIR, "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH,
                        REPO_URL, CODE_DIR], check=True)
elif CODE_SOURCE == "zip":
    from google.colab import files
    uploaded = files.upload()  # a zip of the repo containing training.py, pilot.py, loss/, model_define/, cv/
    zip_name = next(iter(uploaded))
    shutil.rmtree(CODE_DIR, ignore_errors=True)
    with zipfile.ZipFile(zip_name) as z:
        z.extractall(CODE_DIR)
    # accept a zip whose content sits inside one top-level folder
    entries = os.listdir(CODE_DIR)
    if "training.py" not in entries and len(entries) == 1:
        inner = os.path.join(CODE_DIR, entries[0])
        for name in os.listdir(inner):
            shutil.move(os.path.join(inner, name), CODE_DIR)
        os.rmdir(inner)
else:
    raise ValueError(CODE_SOURCE)

for f in ["training.py", "pilot.py", "loss/partial_loss.py"]:
    assert os.path.exists(os.path.join(CODE_DIR, f)), "missing " + f + ", is the branch / zip up to date?"

# keep results and models on Drive so they survive a disconnect
for name in ["result", "model"]:
    path = os.path.join(CODE_DIR, name)
    if os.path.islink(path):
        continue
    shutil.rmtree(path, ignore_errors=True)
    os.symlink(os.path.join(DRIVE_DIR, name), path)

%cd {CODE_DIR}
!git log --oneline -1 2>/dev/null || true

## 4. Run the pilot

About 6 × 3 × 30 epochs; on a T4 expect a few hours. Output streams below.

In [ ]:
losses = " ".join(LOSSES)
lrs = " ".join(LRS)
!PYTHONUNBUFFERED=1 python pilot.py --dataset {DATASET} --opt_alg {OPT_ALG} --losses {losses} --lrs {lrs} --runs {RUNS} --epoch {EPOCH} --alpha {ALPHA}

## 5. Results

All numbers are accuracies in %. Each run's test accuracy is taken at its best validation epoch; tables and figures are also saved to `MyDrive/label_mask_loss_runs/result`.

### 5.1 Learning-rate sweep

Mean over runs. The highlighted row of each loss is its lr chosen by **validation** accuracy.

In [ ]:
import os
import numpy as np
import pandas as pd
from IPython.display import display

PREFIX = "pilot_{}_{}_a{}".format(DATASET, OPT_ALG, ALPHA)

def run_dir(loss, lr):
    return os.path.join("result", "{}_{}_{}_pilot_a{}_lr{}_result".format(DATASET, OPT_ALG, loss, ALPHA, lr))

def load_summary(loss, lr):
    path = os.path.join(run_dir(loss, lr), "summary.csv")
    return pd.read_csv(path).head(RUNS) if os.path.exists(path) else None

rows = []
for loss in LOSSES:
    for lr in LRS:
        s = load_summary(loss, lr)
        if s is None:
            print("missing: {} lr={}".format(loss, lr))
            continue
        rows.append({"loss": loss, "lr": lr, "runs": len(s),
                     "val_acc": s["best_val_acc"].mean(), "test_acc": s["test_acc"].mean(),
                     "test_std": s["test_acc"].std(), "best_epoch": s["best_epoch"].mean()})
sweep = pd.DataFrame(rows)
sweep["selected"] = sweep.index.isin(sweep.groupby("loss")["val_acc"].idxmax())
chosen = dict(sweep[sweep["selected"]][["loss", "lr"]].values)

display(sweep.style.format(precision=2)
        .apply(lambda r: ["font-weight: bold; background-color: #fff3bf" if r["selected"] else ""] * len(r), axis=1)
        .hide(axis="index"))
sweep.to_csv(os.path.join("result", PREFIX + "_sweep.csv"), index=False)

### 5.2 Comparison at the selected lr

Runs share the same data split and seed sequence, so run *i* of every loss is paired. `wins` counts runs where the loss beats the baseline (first entry of `LOSSES`); `p` is a paired t-test. With only a few runs, p-values are a rough signal.

In [ ]:
from scipy import stats

per_run = pd.DataFrame({loss: load_summary(loss, lr).set_index("run")["test_acc"] for loss, lr in chosen.items()})
per_run.index.name = "run"
base = LOSSES[0]

final = []
for loss in per_run.columns:
    diff = per_run[loss] - per_run[base]
    p = stats.ttest_rel(per_run[loss], per_run[base]).pvalue if loss != base and len(diff) > 1 else np.nan
    final.append({"loss": loss, "lr": chosen[loss], "runs": len(diff),
                  "test_acc": "{:.2f} ± {:.2f}".format(per_run[loss].mean(), per_run[loss].std()),
                  "Δ vs " + base: diff.mean(), "wins": "{}/{}".format((diff > 0).sum(), len(diff)), "p": p})
final = pd.DataFrame(final)
display(final.style.format(precision=2, na_rep="").format({"p": "{:.3f}"}, na_rep="").hide(axis="index"))

print("Test accuracy per run:")
display(per_run.assign(**{loss + " - " + base: per_run[loss] - per_run[base] for loss in per_run.columns if loss != base})
        .style.format(precision=2))

for _, row in final.iterrows():
    if row["loss"] == base:
        continue
    delta = row["Δ vs " + base]
    if np.isnan(row["p"]):
        verdict = "not enough runs for a test"
    elif row["p"] < 0.05:
        verdict = "significantly {} than {}".format("better" if delta > 0 else "worse", base)
    else:
        verdict = "no significant difference from {} (more runs may be needed)".format(base)
    print("{}: Δ = {:+.2f}, {}".format(row["loss"], delta, verdict))

final.to_csv(os.path.join("result", PREFIX + "_final.csv"), index=False)
per_run.to_csv(os.path.join("result", PREFIX + "_per_run.csv"))

### 5.3 Learning curves at the selected lr

Mean over runs; the band is ± one standard deviation.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for loss, lr in chosen.items():
    runs = [pd.read_csv(os.path.join(run_dir(loss, lr), "{}.csv".format(t))) for t in range(RUNS)
            if os.path.exists(os.path.join(run_dir(loss, lr), "{}.csv".format(t)))]
    curves = pd.concat(runs).groupby("epoch")
    mean, std = curves.mean(), curves.std().fillna(0)
    for ax, col in zip(axes, ["val_acc", "test_acc", "training_loss"]):
        ax.plot(mean.index, mean[col], label="{} (lr={})".format(loss, lr))
        ax.fill_between(mean.index, mean[col] - std[col], mean[col] + std[col], alpha=0.2)
for ax, title in zip(axes, ["Validation accuracy (%)", "Test accuracy (%)", "Training loss (sum over batches)"]):
    ax.set_title(title)
    ax.set_xlabel("epoch")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(alpha=0.3)
axes[0].legend()
fig.suptitle("{} / {} / alpha={}".format(DATASET, OPT_ALG, ALPHA))
fig.tight_layout()
fig.savefig(os.path.join("result", PREFIX + "_curves.png"), dpi=150)
plt.show()

## 6. Download results (optional)

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive("/content/pilot_results", "zip", os.path.join(DRIVE_DIR, "result"))
files.download(archive)